# Alpamayo 1.5: Local Offline Inference Demo

This notebook runs Alpamayo 1.5 on a local offline clip directory and performs offline inference.

This version:
- reuses clip-level cache from `load_offline_dataset.py`
- uses shared evaluation helpers from `offline_eval_utils.py`
- keeps the current validated coordinate handling:
  - xyz in raw t0-local frame
  - rot in model/action rotation frame
  - notebook eval applies `EVAL_XY_ROTATION_DEG = -90.0`


In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('cwd =', Path.cwd().resolve())
print('repo_root =', repo_root)
print('src_path =', src_path)
print('src exists =', src_path.exists())
print('PYTHONPATH =', os.environ.get('PYTHONPATH'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import (
    load_offline_dataset,
    get_or_build_offline_clip_cache,
)
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.offline_eval_utils import (
    set_reproducible_seeds,
    history_consistency_table,
    segment_mean_speed_table,
    run_offline_inference_window,
    _compute_adaptive_xy_limits,
)


In [ ]:
# ===== Reproducibility =====
SEED = 42
set_reproducible_seeds(SEED)
print('SEED =', SEED)


In [ ]:
# ===== Local paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Inference config =====
DEVICE = 'cuda'
T0_SOD = 175500.23
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)

# Confirmed eval-side xy rotation for current validated pipeline
EVAL_XY_ROTATION_DEG = -90.0

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('CLIP_DIR =', CLIP_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('PROCESSOR_PATH =', PROCESSOR_PATH)
print('COSMOS_PROCESSOR_PATH =', COSMOS_PROCESSOR_PATH)
print('ALPAMAYO_VLM_PROCESSOR_PATH =', os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'])
print('T0_SOD =', T0_SOD)
print('IMAGE_SIZE =', IMAGE_SIZE)
print('EVAL_XY_ROTATION_DEG =', EVAL_XY_ROTATION_DEG)


### Path validation


In [ ]:
def _describe_path(p: Path, name: str):
    print(f'{name}: {p}')
    print(f'  exists: {p.exists()}')
    print(f'  is_dir: {p.is_dir() if p.exists() else "N/A"}')

_describe_path(CLIP_DIR, 'CLIP_DIR')
_describe_path(MODEL_PATH, 'MODEL_PATH')
_describe_path(PROCESSOR_PATH, 'PROCESSOR_PATH')
_describe_path(COSMOS_PROCESSOR_PATH, 'COSMOS_PROCESSOR_PATH')

if not CLIP_DIR.exists():
    raise FileNotFoundError(f'CLIP_DIR does not exist: {CLIP_DIR}')
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'MODEL_PATH does not exist: {MODEL_PATH}')
if not PROCESSOR_PATH.exists():
    raise FileNotFoundError(f'PROCESSOR_PATH does not exist: {PROCESSOR_PATH}')
if not COSMOS_PROCESSOR_PATH.exists():
    raise FileNotFoundError(f'COSMOS_PROCESSOR_PATH does not exist: {COSMOS_PROCESSOR_PATH}')
if not (MODEL_PATH / 'config.json').exists():
    raise FileNotFoundError(f'Missing config.json under MODEL_PATH: {MODEL_PATH}')


### Build clip cache once


In [ ]:
clip_cache = get_or_build_offline_clip_cache(CLIP_DIR, debug=True, force_rebuild=False)
print('clip cache ready')
print('num cameras =', len(clip_cache.camera_paths))
print('num pose samples =', len(clip_cache.pose_times_sod))


### Load model and processor


In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Model and processor loaded.')


### Load local offline data


In [ ]:
data = load_offline_dataset(
    clip_dir=CLIP_DIR,
    t0_sod=T0_SOD,
    num_history_steps=NUM_HISTORY_STEPS,
    num_future_steps=NUM_FUTURE_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    debug=True,
    image_size=IMAGE_SIZE,
)

print('Offline dataset loaded.')
print('camera_indices:', data['camera_indices'].tolist())
print('image_frames shape:', tuple(data['image_frames'].shape))
print('ego_history_xyz shape:', tuple(data['ego_history_xyz'].shape))
print('ego_history_rot shape:', tuple(data['ego_history_rot'].shape))
print('ego_future_xyz shape:', tuple(data['ego_future_xyz'].shape))
print('ego_future_rot shape:', tuple(data['ego_future_rot'].shape))
print('requested frame indices:', data['requested_video_frame_indices'][0].tolist())
print('actual frame indices (first camera):', data['actual_video_frame_indices'][0].tolist())
print('num_clipped_frames_total:', data['num_clipped_frames_total'])
print('coordinate_frame_summary:', data['coordinate_frame_summary'])


### Time alignment diagnostics


In [ ]:
summary = data['time_alignment_summary']
df_time_alignment = pd.DataFrame([summary])

print('[Time alignment summary]')
display(df_time_alignment)

camera_indices = data['camera_indices'].cpu().tolist()
camera_names = [helper.CAMERA_DISPLAY_NAMES.get(i, f'Camera {i}') for i in camera_indices]

requested_ts = data['requested_image_timestamps_sod'].cpu().numpy()
requested_idx = data['requested_video_frame_indices'].cpu().numpy()
actual_idx = data['actual_video_frame_indices'].cpu().numpy()
clipped_mask = data['clipped_frame_mask'].cpu().numpy()
num_clipped = data['num_clipped_frames_per_camera'].cpu().numpy()
video_num_frames = data['video_num_frames'].cpu().numpy()

rows = []
for cam_row, cam_name in enumerate(camera_names):
    rows.append({
        'camera_index': camera_indices[cam_row],
        'camera_name': cam_name,
        'video_num_frames': int(video_num_frames[cam_row]),
        'requested_timestamps_sod': requested_ts[cam_row].tolist(),
        'requested_frame_indices': requested_idx[cam_row].tolist(),
        'actual_frame_indices': actual_idx[cam_row].tolist(),
        'clipped_frame_mask': clipped_mask[cam_row].tolist(),
        'num_clipped_frames': int(num_clipped[cam_row]),
    })

df_camera_time_diag = pd.DataFrame(rows)
print('[Per-camera frame alignment diagnostics]')
display(df_camera_time_diag)


### Run offline inference window


In [ ]:
result = run_offline_inference_window(
    data=data,
    model=model,
    processor=processor,
    device=DEVICE,
    top_p=TOP_P,
    temperature=TEMPERATURE,
    num_traj_samples=NUM_TRAJ_SAMPLES,
    max_generation_length=MAX_GENERATION_LENGTH,
    time_step=TIME_STEP,
    eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
)

print('Inference done.')


### Diagnose history motion / yaw consistency


In [ ]:
df_hist_consistency = history_consistency_table(
    result['hist_xyz_raw'],
    result['hist_rot'],
    TIME_STEP,
)

print('[History consistency summary]')
display(pd.DataFrame([{
    'mean_abs_yaw_minus_dxy_deg': float(df_hist_consistency['abs_yaw_minus_dxy_deg'].mean()),
    'median_abs_yaw_minus_dxy_deg': float(df_hist_consistency['abs_yaw_minus_dxy_deg'].median()),
    'last3_mean_abs_yaw_minus_dxy_deg': float(df_hist_consistency['abs_yaw_minus_dxy_deg'].tail(3).mean()),
    'last5_mean_abs_yaw_minus_dxy_deg': float(df_hist_consistency['abs_yaw_minus_dxy_deg'].tail(5).mean()),
}]))

print('\n[History consistency tail]')
display(df_hist_consistency.tail(5))


### Show all sampled images used for inference


In [ ]:
image_frames = data['image_frames'].cpu().numpy()
camera_indices = data['camera_indices'].cpu().tolist()
requested_indices = data['requested_video_frame_indices'].cpu().numpy()
actual_indices = data['actual_video_frame_indices'].cpu().numpy()
clipped_mask = data['clipped_frame_mask'].cpu().numpy()

num_cams, num_frames, _, _, _ = image_frames.shape
camera_names = [helper.CAMERA_DISPLAY_NAMES.get(i, f'Camera {i}') for i in camera_indices]

fig, axes = plt.subplots(num_cams, num_frames, figsize=(4 * num_frames, 3.8 * num_cams))
if num_cams == 1 and num_frames == 1:
    axes = np.array([[axes]])
elif num_cams == 1:
    axes = np.array([axes])
elif num_frames == 1:
    axes = np.array([[ax] for ax in axes])

for cam_idx in range(num_cams):
    for t_idx in range(num_frames):
        ax = axes[cam_idx, t_idx]
        frame = np.transpose(image_frames[cam_idx, t_idx], (1, 2, 0))
        ax.imshow(frame)
        ax.axis('off')
        clip_tag = 'CLIPPED' if clipped_mask[cam_idx, t_idx] else 'ok'
        ax.set_title(
            f'{camera_names[cam_idx]}\n'
            f't={t_idx}, req={requested_indices[cam_idx, t_idx]}, '
            f'actual={actual_indices[cam_idx, t_idx]} ({clip_tag})'
        )

plt.tight_layout()
plt.show()


### Show CoT and compute corrected metrics


In [ ]:
print('Chain-of-Causation:')
print(result['cot_value'])

print('[Main metrics - corrected history/GT plotting frame]')
display(result['df_metrics'])


### Segment speed profile


In [ ]:
df_speed_profile = segment_mean_speed_table(
    gt_xyz=result['gt_xyz_plot'],
    model_xyz=result['pred_best_xyz'],
    dt=TIME_STEP,
    segment_sec=1.0,
)

print('[Segment speed profile - corrected plotting frame]')
display(df_speed_profile)


### Plot final trajectory comparison


In [ ]:
xlim, ylim = _compute_adaptive_xy_limits(
    result['hist_xyz_plot'],
    result['gt_xyz_plot'],
    result['pred_best_xyz'],
    min_span=20.0,
    pad_ratio=0.12,
    pad_min=2.0,
)

plt.figure(figsize=(7, 7))
plt.plot(result['hist_xyz_plot'][:, 0], result['hist_xyz_plot'][:, 1], 'b-o', linewidth=2, markersize=3, label='History')
plt.plot(result['gt_xyz_plot'][:, 0], result['gt_xyz_plot'][:, 1], 'k-o', linewidth=2.5, markersize=3.5, label='GT')
plt.plot(result['pred_best_xyz'][:, 0], result['pred_best_xyz'][:, 1], 'r-o', linewidth=2.5, markersize=3.5, label='Pred best')
plt.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f"Offline inference @ t0_sod={T0_SOD}, minADE={float(result['df_metrics'].iloc[0]['minADE_m']):.3f}m")
plt.xlim(*xlim)
plt.ylim(*ylim)
plt.legend()
plt.grid(True, alpha=0.3)
plt.gca().set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()
